In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float

from nnsight import LanguageModel

import sys

sys.path.append("../../../")

from nngine import alter, restore

device = "cuda"

In [2]:
import torch as t
from torch import Tensor
from jaxtyping import Float

device = "cuda"

model = LanguageModel(
    'openai-community/gpt2',
    device_map="cuda:0",
    dispatch=True,
)

# model.tokenizer.padding_side = "right"
# model.tokenizer.truncation_side = "right"

alter(model)

In [3]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (embed): Embedding(50257, 768)
    (pos_embed): Embedding(1024, 768)
    (dropout): Dropout(p=0.1, inplace=False)
    (layers): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
          (qkv): Placeholder
          (q): Placeholder
          (k): Placeholder
          (v): Placeholder
          (split_q): Placeholder
          (split_k): Placeholder
          (split_v): Placeholder
          (heads): Placeholder
          (attn_result): Placeholder
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1,

In [4]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [5]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    with model.trace(clean_dataset.toks):
        clean_logits = model.output.logits.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.logits.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 3.1688125133514404, Corrupt direction: 1.9478503465652466
Clean metric: 0.9999999403953552, Corrupt metric: 0.0


In [6]:
import eap

import importlib

importlib.reload(eap)

graph = eap.EAP(model.config, components=["head", "mlp"], device="cuda")

In [7]:
graph.run(
    model,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)

In [8]:
edges = graph.top_edges(n=20, format=True)

head.10.7 -> [0.039] -> head.11.10.q
head.9.9 -> [-0.036] -> head.11.10.q
mlp.0 -> [0.026] -> head.6.9.q
head.5.5 -> [0.025] -> head.8.6.v
head.9.9 -> [-0.02] -> head.10.7.q
head.5.5 -> [-0.018] -> head.6.9.q
head.5.5 -> [-0.018] -> mlp.5
head.9.6 -> [-0.017] -> head.11.10.q
mlp.0 -> [-0.017] -> head.11.10.k
mlp.0 -> [-0.016] -> mlp.4
head.4.11 -> [0.016] -> head.6.9.k
head.5.5 -> [-0.015] -> mlp.6
head.4.11 -> [-0.014] -> mlp.5
head.3.0 -> [-0.013] -> mlp.5
mlp.0 -> [-0.013] -> mlp.5
mlp.0 -> [-0.013] -> head.10.7.k
head.9.6 -> [-0.011] -> head.10.7.q
head.10.10 -> [-0.011] -> head.11.10.q
head.2.2 -> [0.011] -> head.6.9.k
mlp.5 -> [-0.011] -> mlp.6
